# Agent 5: Macroeconomic Stress Testing

## Context
Regulatory bodies require financial institutions to undergo Stress Testing to prove they can survive severe economic shocks. The **Stress Testing Agent** simulates predefined crisis scenarios against the portfolio.

## Working Logic
1. **Scenario Definition:** Uses multipliers to define how different shocks affect Claims, Market Risk, and CAT Risk.
2. **Shock Application:** Applies these multipliers to the base portfolio. Catastrophe shocks are based on `Exposure_At_Risk`.
3. **Model & Credit Penalty:** Adjusts operational and credit risks based on underlying ML model performance (AUC/PSI) and reinsurer ratings.
4. **Capital Consumption & Solvency:** Calculates required capital and Stressed Solvency Ratio.

## Key Concepts
- **Macroeconomic Shock:** Sudden, unexpected events that dramatically impact the economy.
- **Exposure At Risk:** The total sum insured that is vulnerable to a tail event.

In [ ]:
import pandas as pd
import numpy as np

# Load data
try:
    df = pd.read_excel('Dataset/Cleaned.xlsx')
except:
    df = pd.DataFrame({
        "Claim_Amount": np.random.uniform(5000, 15000, 100),
        "Written_Premium": np.random.uniform(10000, 20000, 100),
        "Exposure_At_Risk": np.random.uniform(100000, 200000, 100),
        "Capital_Reserve": [5000000] * 100,
        "Insurance_Risk": np.random.uniform(1, 10, 100),
        "Market_Risk": np.random.uniform(1, 10, 100),
        "Credit_Risk": np.random.uniform(1, 10, 100),
        "Operational_Risk": np.random.uniform(1, 10, 100),
        "Hazard_Score": np.random.uniform(1, 10, 100),
        "Fraud_Flag": np.random.choice([0, 1], 100, p=[0.9, 0.1]),
        "Reinsurer_Rating": np.random.choice(['AAA', 'AA', 'A', 'BBB'], 100),
        "AUC": [0.85] * 100,
        "PSI": [0.05] * 100
    })
print("Data loaded with shape:", df.shape)

### 1. Extract Portfolio Baseline

In [ ]:
# Aggregate to portfolio level
portfolio = {
    "claim_amount": df["Claim_Amount"].sum(),
    "written_premium": df["Written_Premium"].sum(),
    "capital_reserve": df["Capital_Reserve"].mean() if "Capital_Reserve" in df.columns else 50000000,
    "exposure_at_risk": df["Exposure_At_Risk"].sum(),
    "insurance_risk": df["Insurance_Risk"].mean() * 10,
    "market_risk": df["Market_Risk"].mean() * 10,
    "credit_risk": df["Credit_Risk"].mean() * 10,
    "operational_risk": df["Operational_Risk"].mean() * 10,
    "catastrophe_risk": df["Hazard_Score"].mean() * 10,
    "fraud_flag": df["Fraud_Flag"].mean(),
    "auc": df["AUC"].mean() if "AUC" in df.columns else 0.85,
    "psi": df["PSI"].mean() if "PSI" in df.columns else 0.05
}

# Reinsurer Rating (Map string to numeric)
rating_map = {'AAA': 1.0, 'AA': 0.9, 'A': 0.8, 'BBB': 0.6, 'BB': 0.4, 'B': 0.2}
if "Reinsurer_Rating" in df.columns:
    portfolio["reinsurer_rating"] = df["Reinsurer_Rating"].map(rating_map).fillna(1.0).mean()
else:
    portfolio["reinsurer_rating"] = 1.0

print("Base Premium:", portfolio["written_premium"])
print("Exposure at Risk:", portfolio["exposure_at_risk"])

### 2. Apply Shock Scenario (e.g., Extreme Combined Shock)

In [ ]:
scenario_mults = {
    "claim_mult": 1.50, # Claims jump 50%
    "mkt_mult": 3.0,    # Market volatility 3x
    "cat_mult": 5.0     # CAT events 5x worse
}

p, c, exp = portfolio["written_premium"], portfolio["capital_reserve"], portfolio["exposure_at_risk"]

stressed_claim = portfolio["claim_amount"] * scenario_mults["claim_mult"]
stressed_mkt = min(100, portfolio["market_risk"] * scenario_mults["mkt_mult"])
stressed_cat = min(100, portfolio["catastrophe_risk"] * scenario_mults["cat_mult"])

stressed_lr = (stressed_claim / p * 100) if p > 0 else 0
claim_impact = max(0.0, -(p - stressed_claim))
market_impact = (stressed_mkt / 100) * c * 0.40

# Catastrophe impact based on Exposure At Risk
cat_impact = (stressed_cat / 100) * exp * 0.05 * scenario_mults["cat_mult"]

# Credit Shock with Reinsurer Dampener
reinsurer_dampener = 1.0 + (1.0 - portfolio["reinsurer_rating"]) * 0.5 
credit_impact = (portfolio["credit_risk"] / 100) * c * 0.20 * reinsurer_dampener

# Operational Shock with Fraud & Model Penalty
fraud_penalty = 1.0 + portfolio["fraud_flag"] * 0.5
model_penalty = 1.25 if portfolio["auc"] < 0.70 or portfolio["psi"] > 0.1 else 1.0
op_impact = (portfolio["operational_risk"] / 100) * c * 0.15 * fraud_penalty * model_penalty

total_consumed = claim_impact + market_impact + cat_impact + credit_impact + op_impact
solvency_ratio = ((c - total_consumed) / c * 100)

print(f"Stressed Loss Ratio: {stressed_lr:.2f}%")
print(f"Capital Consumed: ${total_consumed:,.2f}")
print(f"Stressed Solvency Ratio: {solvency_ratio:.2f}%")
if solvency_ratio < 100:
    print("WARNING: Capital Shortfall Triggered!")
else:
    print("Solvent under stress.")